<a href="https://colab.research.google.com/github/SergeiLab/video-recommendation-system/blob/main/recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =============================================================================
# ПРОСТАЯ РЕКОМЕНДАТЕЛЬНАЯ СИСТЕМА - LIGHTGBM
# Работает быстро и стабильно на Google Colab
# =============================================================================

# Установка
print("Installing...")
!pip install -q lightgbm

import pandas as pd
import numpy as np
import pickle
import lightgbm as lgbm
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print(f"SEED: {SEED}\n")

# =============================================================================
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# =============================================================================

print("Loading data...")
train = pd.read_csv('interactions_train.csv')
users = pd.read_csv('users.csv')
items = pd.read_csv('items.csv')

print(f"Train: {train.shape}, Users: {users.shape}, Items: {items.shape}\n")

# Merge
train = train.merge(users, on='user_id', how='left')
train = train.merge(items, on='item_id', how='left')

# Target
train['target'] = (train['watched_pct'] > 50).astype(int)
print(f"Target: {train['target'].value_counts(normalize=True).to_dict()}\n")

# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

print("Feature engineering...")

def create_features(df):
    df = df.copy()

    # Temporal
    df['last_watch_dt'] = pd.to_datetime(df['last_watch_dt'])
    df['hour'] = df['last_watch_dt'].dt.hour
    df['dow'] = df['last_watch_dt'].dt.dayofweek
    df['month'] = df['last_watch_dt'].dt.month

    # Duration
    df['dur_min'] = df['total_dur'] / 60
    df['dur_log'] = np.log1p(df['total_dur'])

    # Release year
    df['release_year'] = df['release_year'].fillna(2020)
    df['years_old'] = 2024 - df['release_year']

    # Genres
    df['n_genres'] = df['genres'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)

    # Countries
    df['n_countries'] = df['countries'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
    df['is_ru'] = df['countries'].fillna('').str.contains('Россия|СССР', case=False).astype(int)
    df['is_usa'] = df['countries'].fillna('').str.contains('США', case=False).astype(int)

    # Cast
    df['n_actors'] = df['actors'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
    df['n_directors'] = df['directors'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)

    return df

train = create_features(train)

# Encode categoricals
cat_cols = ['age', 'income', 'sex', 'content_type', 'for_kids', 'age_rating']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = train[col].fillna('unknown').astype(str)
    train[col + '_enc'] = le.fit_transform(train[col])
    encoders[col] = le

# Encode IDs for embeddings later
user_enc = LabelEncoder()
item_enc = LabelEncoder()
train['user_idx'] = user_enc.fit_transform(train['user_id'])
train['item_idx'] = item_enc.fit_transform(train['item_id'])

print(f"Users: {train['user_idx'].nunique()}, Items: {train['item_idx'].nunique()}\n")

# User statistics
user_stats = train.groupby('user_id').agg({
    'watched_pct': ['mean', 'std', 'count'],
    'total_dur': 'mean'
}).reset_index()
user_stats.columns = ['user_id', 'user_avg_watch', 'user_std_watch', 'user_count', 'user_avg_dur']

# Item statistics
item_stats = train.groupby('item_id').agg({
    'watched_pct': ['mean', 'std', 'count']
}).reset_index()
item_stats.columns = ['item_id', 'item_avg_watch', 'item_std_watch', 'item_count']

train = train.merge(user_stats, on='user_id', how='left')
train = train.merge(item_stats, on='item_id', how='left')

# Features
features = [
    'hour', 'dow', 'month', 'dur_min', 'dur_log', 'years_old',
    'n_genres', 'n_countries', 'is_ru', 'is_usa', 'n_actors', 'n_directors',
    'kids_flg', 'user_avg_watch', 'user_std_watch', 'user_count', 'user_avg_dur',
    'item_avg_watch', 'item_std_watch', 'item_count'
] + [c + '_enc' for c in cat_cols]

for f in features:
    train[f] = train[f].fillna(0)

X = train[features]
y = train['target']

print(f"Features: {len(features)}\n")

# =============================================================================
# ОБУЧЕНИЕ МОДЕЛИ
# =============================================================================

print("Training LightGBM...\n")

model = lgbm.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

model.fit(X, y)

# Feature importance
fi = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 features:")
print(fi.head(15))
print()

# Save
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(user_enc, open('user_enc.pkl', 'wb'))
pickle.dump(item_enc, open('item_enc.pkl', 'wb'))
pickle.dump(encoders, open('cat_enc.pkl', 'wb'))
pickle.dump(user_stats, open('user_stats.pkl', 'wb'))
pickle.dump(item_stats, open('item_stats.pkl', 'wb'))

print("Model saved!\n")

# =============================================================================
# ПРЕДСКАЗАНИЕ
# =============================================================================

print("="*80)
print("PREDICTION")
print("="*80 + "\n")

# Load test
test_users = pd.read_csv('users_public_test.csv')
test_users = test_users.merge(users, on='user_id', how='left')

# Prepare items
items['release_year'] = items['release_year'].fillna(2020)
items['years_old'] = 2024 - items['release_year']
items['n_genres'] = items['genres'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
items['n_countries'] = items['countries'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
items['is_ru'] = items['countries'].fillna('').str.contains('Россия|СССР', case=False).astype(int)
items['is_usa'] = items['countries'].fillna('').str.contains('США', case=False).astype(int)
items['n_actors'] = items['actors'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
items['n_directors'] = items['directors'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)

# Encode test users
for col in cat_cols:
    if col in test_users.columns:
        test_users[col] = test_users[col].fillna('unknown').astype(str)
        test_users[col + '_enc'] = test_users[col].apply(
            lambda x: encoders[col].transform([x])[0] if x in encoders[col].classes_ else 0
        )

# Encode items
for col in cat_cols:
    if col in items.columns:
        items[col] = items[col].fillna('unknown').astype(str)
        items[col + '_enc'] = items[col].apply(
            lambda x: encoders[col].transform([x])[0] if x in encoders[col].classes_ else 0
        )

# History
history = train.groupby('user_id')['item_id'].apply(set).to_dict()

# Popular fallback
popular = train.groupby('item_id')['watched_pct'].mean().nlargest(20).index.tolist()

# Known items only
known_items = items[items['item_id'].isin(item_enc.classes_)].copy()
known_items['item_idx'] = item_enc.transform(known_items['item_id'])

print(f"Test users: {len(test_users)}")
print(f"Known items: {len(known_items)}\n")

# Recommend
print("Generating recommendations...\n")
results = []

for _, user in tqdm(test_users.iterrows(), total=len(test_users)):
    uid = user['user_id']

    # Cold start
    if uid not in user_enc.classes_:
        results.append([uid] + popular[:10])
        continue

    watched = history.get(uid, set())
    candidates = known_items[~known_items['item_id'].isin(watched)]

    if len(candidates) == 0:
        results.append([uid] + popular[:10])
        continue

    # Sample for speed
    if len(candidates) > 5000:
        pop_cands = candidates[candidates['item_id'].isin(popular[:1000])]
        other_cands = candidates[~candidates['item_id'].isin(popular[:1000])].sample(
            min(4000, len(candidates) - len(pop_cands)), random_state=SEED
        )
        candidates = pd.concat([pop_cands, other_cands])

    # Prepare features
    cand_df = candidates.copy()

    # Add user features
    for col in ['age', 'income', 'sex', 'kids_flg']:
        if col + '_enc' in user.index:
            cand_df[col + '_enc'] = user[col + '_enc']
        elif col in user.index:
            cand_df[col] = user[col]

    # Add user stats
    user_stat = user_stats[user_stats['user_id'] == uid]
    if len(user_stat) > 0:
        for col in ['user_avg_watch', 'user_std_watch', 'user_count', 'user_avg_dur']:
            cand_df[col] = user_stat[col].values[0]
    else:
        for col in ['user_avg_watch', 'user_std_watch', 'user_count', 'user_avg_dur']:
            cand_df[col] = 0

    # Add item stats
    cand_df = cand_df.merge(item_stats, on='item_id', how='left')

    # Temporal features (use median from training)
    cand_df['hour'] = 19
    cand_df['dow'] = 5
    cand_df['month'] = 10
    cand_df['dur_min'] = 90
    cand_df['dur_log'] = np.log1p(90 * 60)

    # Fill missing
    for f in features:
        if f not in cand_df.columns:
            cand_df[f] = 0
        cand_df[f] = cand_df[f].fillna(0)

    # Predict
    X_cand = cand_df[features]
    scores = model.predict_proba(X_cand)[:, 1]

    # Top 10
    top_idx = np.argsort(scores)[-10:][::-1]
    top_items = cand_df.iloc[top_idx]['item_id'].tolist()

    # Fill if needed
    if len(top_items) < 10:
        for p in popular:
            if p not in top_items and p not in watched:
                top_items.append(p)
            if len(top_items) == 10:
                break

    results.append([uid] + top_items[:10])

# Save
sub = pd.DataFrame(results, columns=['user_id'] + list(range(10)))
sub.to_csv(f'submission_{SEED}.csv', index=False)

print(f"\n✓ Saved: submission_{SEED}.csv")
print(f"Shape: {sub.shape}\n")
print("Sample:")
print(sub.head())

# Download
print("\n" + "="*80)
print("DOWNLOADING...")
print("="*80)
from google.colab import files
files.download(f'submission_{SEED}.csv')

print("\n✓ DONE!")

Installing...
SEED: 42

Loading data...
Train: (922967, 5), Users: (840197, 5), Items: (15963, 13)

Target: {0: 0.5333917680697143, 1: 0.4666082319302857}

Feature engineering...
Users: 284999, Items: 11856

Features: 26

Training LightGBM...

Top 15 features:
             feature  importance
3            dur_min        2458
19        item_count        1731
17    item_avg_watch        1613
18    item_std_watch        1247
13    user_avg_watch        1032
5          years_old         991
14    user_std_watch         942
10          n_actors         918
16      user_avg_dur         680
4            dur_log         613
15        user_count         570
6           n_genres         461
25    age_rating_enc         458
11       n_directors         336
23  content_type_enc         291

Model saved!

PREDICTION

Test users: 198636
Known items: 11856

Generating recommendations...



100%|██████████| 198636/198636 [3:10:25<00:00, 17.38it/s]



✓ Saved: submission_42.csv
Shape: (198636, 11)

Sample:
   user_id      0     1     2     3      4      5      6     7      8     9
0        4  15653  5747   161  2946   4145  12004    224  8561   4881  8205
1        8      6    15    36    49     73    100    136   145    146   158
2       10      6    15    36    49     73    100    136   145    146   158
3       15      6    15    36    49     73    100    136   145    146   158
4       16  15653  4548  5944    36  12004    224  13859  4145  11644  1355

DOWNLOADING...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ DONE!
